## From Shunans scripts

In [ ]:
import glob
import vaex as vx
import numpy as np
import cmocean as cm
import matplotlib.pyplot as plt
# import pandas as pd
from scipy import stats
import seaborn as sns
from typing import Any, NamedTuple, Optional
import gc

In [ ]:
class RegressionResult(NamedTuple):
    slope: float
    intercept: float
    rvalue: float
    pvalue: float
    rmse: float


In [ ]:
def linregress_vaex(df: Any, x: str, y: str, selection: Optional[str] = None) -> RegressionResult:
    """Compute linear-regression summary from Vaex aggregations (out-of-core)."""
    base_selection = f"isfinite({x}) & isfinite({y})"
    if selection:
        selection = f"({base_selection}) & ({selection})"
    else:
        selection = base_selection

    n = float(df.count(selection=selection))
    if n < 3:
        raise ValueError("Need at least 3 finite points to compute regression statistics")

    mean_x = float(df.mean(x, selection=selection))
    mean_y = float(df.mean(y, selection=selection))
    var_x = float(df.var(x, selection=selection))
    var_y = float(df.var(y, selection=selection))
    mean_xy = float(df.mean(df[x] * df[y], selection=selection))
    cov_xy = mean_xy - mean_x * mean_y

    if var_x <= 0 or var_y <= 0:
        raise ValueError("Variance is zero; regression is undefined")

    slope = cov_xy / var_x
    intercept = mean_y - slope * mean_x
    rvalue = cov_xy / np.sqrt(var_x * var_y)

    # Numerical guard to keep r within [-1, 1] for p-value computation.
    rvalue = float(np.clip(rvalue, -1.0, 1.0))
    if abs(rvalue) == 1.0:
        pvalue = 0.0
    else:
        t_stat = rvalue * np.sqrt((n - 2.0) / (1.0 - rvalue**2))
        pvalue = float(2.0 * stats.t.sf(abs(t_stat), df=n - 2.0))
    # r2 = df.ml.metrics.r2_score(df[y], df[x], selection=selection)
    
    mse = float(df.mean((df[x] - df[y]) ** 2, selection=selection))
    rmse = float(np.sqrt(mse))
    
    return RegressionResult(
        slope=float(slope),
        intercept=float(intercept),
        rvalue=float(rvalue),
        pvalue=float(pvalue),
        rmse = rmse
    )


#%% training data

df = vx.open("/home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/Output_tables/Training/test_train/*.h5")

#%% statistics between satlst and era5: slope, intercept, r2, and p-value of linear regression
results = linregress_vaex(df, "era5", "satlst")

print(f"slope: {results.slope}")
print(f"intercept: {results.intercept}")
print(f"r2: {results.rvalue**2}")
print(f"p-value: {results.pvalue}")
print(f"RMSE: {results.rmse}")
# print(f"Number of points used in regression: {df.count(selection=selection):,}")
print(f"Number of points in full dataset: {df.count():,}")
print(results)

#%%
range = [-60,30]
fig, ax = plt.subplots(figsize=(8,7))
ax.plot(np.array(range), results.slope * np.array(range) + results.intercept, color='red')
ax.plot(range, range, '--', color='gray', alpha=0.8, label='1:1 line')
ax.set_xlim(range)
ax.set_ylim(range)
tick_values = np.arange(-60, 31, 20)
ax.set_xticks(tick_values)
ax.set_yticks(tick_values)
df.viz.heatmap(
    x="era5", 
    y="satlst", 
    what=np.log(vx.stat.count()),
    show=False,
    # vmin=0, vmax=20,
    colormap = cm.cm.haline
    )


ax.set_aspect('equal')
ax.grid(True, color= 'dimgrey', which='major', linestyle='-', linewidth=0.5, alpha=0.2)
ax.set_title('ERA5 vs. SAT Land Surface Temperature (°C)')
ax.set(xlabel="Downscaled ERA5 LST 1000m (°C)", ylabel="Harmonized Satellite LST 1000m (°C)")
fig.savefig("./CalibrationPlots/regression_plot_land.png", dpi=300)
# close figure after saved
plt.close(fig)


# Close the training df to avoid "Too many files open - errno 24" 
if hasattr(df, "close"):
    df.close()
    del df
    gc.collect() 

             
# TESTING DATA / CALIBRATION
df = vx.open("/home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/Output_tables/Testing/test_test/*.h5")
df["era5_calibrated"] = results.slope * df.era5 + results.intercept

results = linregress_vaex(df, "era5_calibrated", "satlst")
print(f"slope: {results.slope}")
print(f"intercept: {results.intercept}")
print(f"r2: {results.rvalue**2}")
print(f"p-value: {results.pvalue}")
print(f"RMSE: {results.rmse}")
# print(f"Number of points used in regression: {df.count(selection=validation_selection):,}")
print(f"Number of points in full dataset: {df.count():,}")
print(results)

# %%
fig, ax = plt.subplots(figsize=(8,7))
ax.plot(np.array(range), results.slope * np.array(range) + results.intercept, color='red')
ax.plot(range, range, '--', color='gray', alpha=0.8, label='1:1 line')
ax.set_xlim(range)
ax.set_ylim(range)
tick_values = np.arange(-60, 31, 20)
ax.set_xticks(tick_values)
ax.set_yticks(tick_values)
df.viz.heatmap(
    x="era5_calibrated", 
    y="satlst", 
    what=np.log(vx.stat.count()),
    show=False,
    # vmin=0, vmax=20,
    colormap = cm.cm.haline
    )
ax.set_aspect('equal')
ax.grid(True, color= 'dimgrey', which='major', linestyle='-', linewidth=0.5, alpha=0.2)
ax.set_title('Calibrated ERA5 vs. SAT LST (°C)')
ax.set(xlabel="Calibrated ERA5 LST 1000m (°C)", ylabel="Harmonized Satellite LST 1000m (°C)")
fig.savefig("./CalibrationPlots/calibration_plot_land_test.png", dpi=300)
# close figure after saved
plt.close(fig)

df.close()
# %%

## Old parts of the script

In [ ]:
### LINEAR REGRESSION ### 
    # print(f'Image:{date}')

    slope, intercept, r_value, p_value, std_err = linregress(era5_1d_nona, satlst_1d_nona)
    # print(f'Total\tslope: {round(slope, 3)}, intercept: {round(intercept, 3)}, r_value: {round(r_value, 2)}, p_value: {round(p_value,3)}, std_err: {round(std_err, 3)}')

    slope_i, intercept_i, r_value_i, p_value_i, std_err_i = linregress(era5_1d_nona_ice, satlst_1d_nona_ice)
    # print(f'Ice\tslope: {round(slope_i, 3)}, intercept: {round(intercept_i, 3)}, r_value: {round(r_value_i, 2)}, p_value: {round(p_value_i,3)}, std_err: {round(std_err_i, 3)}')

    slope_l, intercept_l, r_value_l, p_value_l, std_err_l = linregress(era5_1d_nona_land, satlst_1d_nona_land)
    # print(f'Land\tslope: {round(slope_l, 3)}, intercept: {round(intercept_l, 3)}, r_value: {round(r_value_l, 2)}, p_value: {round(p_value_l,3)}, std_err: {round(std_err_l, 3)}\n')


### PLOT (for test purposes) 

    fig, axes = plt.subplots(1, 3, figsize=(12, 6))
    xlims = (era5_1d_nona.min(), era5_1d_nona.max())
    ylims = (satlst_1d_nona.min(), satlst_1d_nona.max())
    
    # Plot 1 total
    ax = axes[0]
    hexbin = ax.hexbin(era5_1d_nona, satlst_1d_nona,
                    gridsize=50, cmap=cmocean.cm.haline, bins='log', mincnt=1)
    
    sns.regplot(ax=ax, x=era5_1d_nona, y=satlst_1d_nona, scatter=False, color='red')
    ax.plot([-50, 30], [-50, 30], 'k--', alpha=0.5, label='1:1 line')
    ax.set_title('Total')
    ax.set_aspect('equal')
    ax.set_xlim([-50, 30])
    ax.set_ylim([-50, 30])
    ax.set_xlabel('ERA5 T2M (°C)')
    ax.set_ylabel('SATLST (°C)')
    cb = fig.colorbar(hexbin, ax=ax)
    cb.set_label('log(Count+1)')

    # Add statistics text box for training data
    stats_text = f'N = {era5_1d_nona.shape[0]}\n'
    stats_text += f'R² = {r_value**2:.3f}\n'
    stats_text += f'Slope = {slope:.3f}\n'
    stats_text += f'Intercept = {intercept:.3f}\n'
   
    ax.text(0.50, 0.35, stats_text, transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))


    # Plot 2 Ice
    ax = axes[1]
    hexbin = ax.hexbin(era5_1d_nona_ice, satlst_1d_nona_ice,
                    gridsize=50, cmap=cmocean.cm.haline, bins='log', mincnt=1)
    
    sns.regplot(ax=ax, x=era5_1d_nona_ice, y=satlst_1d_nona_ice, scatter=False, color='red')
    ax.plot([-50, 30], [-50, 30], 'k--', alpha=0.5, label='1:1 line')
    ax.set_title('Ice')
    ax.set_aspect('equal')
    ax.set_xlim([-50, 30])
    ax.set_ylim([-50, 30])
    ax.set_xlabel('ERA5 T2M (°C)')
    ax.set_ylabel('SATLST (°C)')
    cb = fig.colorbar(hexbin, ax=ax)
    cb.set_label('log(Count+1)')

    # Add statistics text box for training data
    stats_text = f'N = {era5_1d_nona_ice.shape[0]}\n'
    stats_text += f'R² = {r_value_i**2:.3f}\n'
    stats_text += f'Slope = {slope_i:.3f}\n'
    stats_text += f'Intercept = {intercept_i:.3f}\n'
   
    ax.text(0.50, 0.35, stats_text, transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    # Plot 3 Land
    ax = axes[2]
    hexbin = ax.hexbin(era5_1d_nona_land, satlst_1d_nona_land,
                    gridsize=50, cmap=cmocean.cm.haline, bins='log', mincnt=1)
    
    sns.regplot(ax=ax, x=era5_1d_nona_land, y=satlst_1d_nona_land, scatter=False, color='red')
    ax.plot([-50, 30], [-50, 30], alpha=0.5, label='1:1 line')
    ax.set_title('Land')
    ax.set_aspect('equal')
    ax.set_xlim([-50, 30])
    ax.set_ylim([-50, 30])
    ax.set_xlabel('ERA5 T2M (°C)')
    ax.set_ylabel('SATLST (°C)')
    cb = fig.colorbar(hexbin, ax=ax)
    cb.set_label('log(Count+1)')

    # Add statistics text box for training data
    stats_text = f'N = {era5_1d_nona_land.shape[0]}\n'
    stats_text += f'R² = {r_value_l**2:.3f}\n'
    stats_text += f'Slope = {slope_l:.3f}\n'
    stats_text += f'Intercept = {intercept_l:.3f}\n'
   
    ax.text(0.50, 0.35, stats_text, transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    fig.suptitle(f'ERA5 vs SATLST on {date}', fontsize=16)
    plt.tight_layout()
    plt.show()